# Semana 10: Series de Tiempo
## Notebook Conceptual (NB1) – Descomposición y Features Temporales

**Propósito:** Modelar datos temporales, descomponer series y aplicar tanto modelos clásicos (ARIMA) como ML con características temporales.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Generar y visualizar series temporales sintéticas con tendencia, estacionalidad y ruido.
- Aplicar descomposición aditiva y multiplicativa con statsmodels.
- Crear lags y ventanas móviles usando pandas.
- Visualizar funciones de autocorrelación (ACF) y autocorrelación parcial (PACF).
- Ajustar un modelo ARIMA manualmente y evaluar su rendimiento.
- Convertir la serie en problema supervisado y usar Random Forest para predicción.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Statsmodels para series temporales
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

# Scikit-learn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Generación de Serie Temporal Sintética

Creamos una serie con:
- Tendencia lineal creciente
- Estacionalidad anual (período 12)
- Ruido aleatorio

Esta serie simula datos mensales con patrón estacional.

In [ ]:
# Parámetros
n_periods = 120  # 10 años de datos mensuales
t = np.arange(n_periods)

# Componentes
trend = 0.1 * t  # tendencia lineal
seasonal = 5 * np.sin(2 * np.pi * t / 12)  # estacionalidad anual
noise = np.random.normal(0, 2, n_periods)  # ruido

# Serie aditiva
y = trend + seasonal + noise

# Creamos un índice de fechas
dates = pd.date_range(start='2015-01-01', periods=n_periods, freq='M')
series = pd.Series(y, index=dates, name='valor')

# Visualizamos
plt.figure(figsize=(12, 5))
plt.plot(series)
plt.title('Serie Temporal Sintética (Tendencia + Estacionalidad + Ruido)')
plt.xlabel('Fecha')
plt.ylabel('Valor')
plt.grid(True)
plt.show()

---
## 2. Descomposición de la Serie Temporal

Usamos `seasonal_decompose` de statsmodels para descomponer la serie en tendencia, estacionalidad y residuo.

Probamos tanto descomposición aditiva como multiplicativa.

In [ ]:
# Descomposición aditiva (asume que los componentes se suman)
decomp_add = seasonal_decompose(series, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10))
series.plot(ax=axes[0], title='Serie Original')
decomp_add.trend.plot(ax=axes[1], title='Tendencia')
decomp_add.seasonal.plot(ax=axes[2], title='Estacionalidad')
decomp_add.resid.plot(ax=axes[3], title='Residuo')
plt.tight_layout()
plt.show()

In [ ]:
# Para comparar, generamos una serie multiplicativa
y_mult = (100 + trend) * (1 + 0.2 * np.sin(2 * np.pi * t / 12)) * (1 + noise/20)
series_mult = pd.Series(y_mult, index=dates, name='valor')

# Descomposición multiplicativa
decomp_mult = seasonal_decompose(series_mult, model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10))
series_mult.plot(ax=axes[0], title='Serie Multiplicativa')
decomp_mult.trend.plot(ax=axes[1], title='Tendencia')
decomp_mult.seasonal.plot(ax=axes[2], title='Estacionalidad')
decomp_mult.resid.plot(ax=axes[3], title='Residuo')
plt.tight_layout()
plt.show()

---
## 3. Creación de Lags y Ventanas Móviles

Usamos pandas para crear características temporales:
- Rezagos (lags) de la serie
- Media móvil
- Desviación estándar móvil

In [ ]:
# Creamos un DataFrame con la serie
df = pd.DataFrame({'y': series})

# Lags (rezagos)
for lag in [1, 2, 3, 6, 12]:
    df[f'lag_{lag}'] = df['y'].shift(lag)

# Ventanas móviles
df['rolling_mean_3'] = df['y'].rolling(window=3).mean()
df['rolling_std_3'] = df['y'].rolling(window=3).std()
df['rolling_mean_6'] = df['y'].rolling(window=6).mean()
df['rolling_mean_12'] = df['y'].rolling(window=12).mean()

# Mostramos las primeras filas
df.head(15)

---
## 4. Visualización de Autocorrelación (ACF) y Autocorrelación Parcial (PACF)

Estas funciones ayudan a identificar el orden de los modelos ARIMA.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

plot_acf(series, lags=30, ax=axes[0])
axes[0].set_title('Autocorrelación (ACF)')

plot_pacf(series, lags=30, ax=axes[1])
axes[1].set_title('Autocorrelación Parcial (PACF)')

plt.tight_layout()
plt.show()

### 4.1. Prueba de Estacionariedad (Dickey-Fuller Aumentado)

Un requisito para ARIMA es que la serie sea estacionaria (media y varianza constantes en el tiempo).

In [ ]:
def test_stationarity(timeseries):
    # Estadístico de Dickey-Fuller
    result = adfuller(timeseries.dropna())
    print('Estadístico ADF:', result[0])
    print('p-valor:', result[1])
    print('Valores críticos:')
    for key, value in result[4].items():
        print(f'\t{key}: {value}')
    
    if result[1] <= 0.05:
        print("\nRechazamos H0: la serie es estacionaria.")
    else:
        print("\nNo rechazamos H0: la serie no es estacionaria.")

print("Prueba de estacionariedad para la serie original:")
test_stationarity(series)

### 4.2. Diferenciación para hacer la serie estacionaria

Aplicamos una diferencia regular para eliminar la tendencia.

In [ ]:
series_diff = series.diff().dropna()

plt.figure(figsize=(12, 4))
plt.plot(series_diff)
plt.title('Serie Diferenciada (1 diferencia)')
plt.grid(True)
plt.show()

print("\nPrueba de estacionariedad para la serie diferenciada:")
test_stationarity(series_diff)

---
## 5. Ajuste de Modelo ARIMA

Ajustamos un modelo ARIMA manualmente. Basado en las gráficas ACF/PACF de la serie diferenciada, elegimos órdenes iniciales.

In [ ]:
# Dividimos en entrenamiento y prueba (últimos 12 meses)
train = series[:-12]
test = series[-12:]

# Ajustamos modelo ARIMA(1,1,1) con estacionalidad (1,1,1,12)
model = ARIMA(train, order=(1,1,1), seasonal_order=(1,1,1,12))
fitted = model.fit()

print(fitted.summary())

# Pronóstico
forecast = fitted.forecast(steps=12)
forecast.index = test.index

# Evaluación
mae_arima = mean_absolute_error(test, forecast)
rmse_arima = np.sqrt(mean_squared_error(test, forecast))
print(f"\nMAE: {mae_arima:.2f}")
print(f"RMSE: {rmse_arima:.2f}")

# Visualización
plt.figure(figsize=(12, 5))
plt.plot(train, label='Entrenamiento')
plt.plot(test, label='Real (test)')
plt.plot(forecast, label='Pronóstico ARIMA', color='red')
plt.title('Pronóstico con ARIMA')
plt.legend()
plt.grid(True)
plt.show()

---
## 6. Conversión a Problema Supervisado y Predicción con Random Forest

Creamos un dataset con lags y características, y usamos Random Forest para predecir.

In [ ]:
# Creamos features con lags hasta 12
df_sup = pd.DataFrame({'y': series})
for lag in range(1, 13):
    df_sup[f'lag_{lag}'] = df_sup['y'].shift(lag)

# Añadimos variables temporales
df_sup['month'] = df_sup.index.month
df_sup['quarter'] = df_sup.index.quarter
df_sup['year'] = df_sup.index.year

# Eliminamos filas con NaN (primeros 12 registros)
df_sup = df_sup.dropna()

# Definimos X e y
X = df_sup.drop('y', axis=1)
y = df_sup['y']

# Dividimos en train/test temporal (últimos 12 como test)
train_size = len(X) - 12
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

print(f"Tamaño entrenamiento: {len(X_train)}")
print(f"Tamaño prueba: {len(X_test)}")

In [ ]:
# Entrenamos Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Predicciones
y_pred_rf = rf.predict(X_test)

# Evaluación
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
print(f"Random Forest - MAE: {mae_rf:.2f}")
print(f"Random Forest - RMSE: {rmse_rf:.2f}")

# Visualización
plt.figure(figsize=(12, 5))
plt.plot(y_train.index, y_train, label='Entrenamiento')
plt.plot(y_test.index, y_test, label='Real (test)')
plt.plot(y_test.index, y_pred_rf, label='Predicción RF', color='red')
plt.title('Predicción con Random Forest (enfoque supervisado)')
plt.legend()
plt.grid(True)
plt.show()

---
## 7. Comparación de Modelos

Comparamos el rendimiento de ARIMA y Random Forest.

In [ ]:
comparacion = pd.DataFrame({
    'Modelo': ['ARIMA(1,1,1)(1,1,1,12)', 'Random Forest'],
    'MAE': [mae_arima, mae_rf],
    'RMSE': [rmse_arima, rmse_rf]
})

print("=== Comparación de Modelos ===")
comparacion.round(4)

In [ ]:
# Importancia de características en Random Forest
importances = rf.feature_importances_
feature_names = X.columns

imp_df = pd.DataFrame({
    'Característica': feature_names,
    'Importancia': importances
}).sort_values('Importancia', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=imp_df, x='Importancia', y='Característica')
plt.title('Importancia de Características - Random Forest')
plt.tight_layout()
plt.show()

---
## 8. Conclusiones

Hemos explorado las técnicas fundamentales para el análisis de series temporales:

✔️ **Generación de serie sintética**: Creamos una serie con tendencia, estacionalidad y ruido.
✔️ **Descomposición**: Separamos la serie en sus componentes usando modelos aditivo y multiplicativo.
✔️ **Features temporales**: Creamos lags y ventanas móviles con pandas.
✔️ **ACF/PACF**: Visualizamos la autocorrelación para identificar órdenes de modelos ARIMA.
✔️ **Estacionariedad**: Aplicamos pruebas de Dickey-Fuller y diferenciación.
✔️ **ARIMA**: Ajustamos un modelo ARIMA estacional y evaluamos su pronóstico.
✔️ **Enfoque supervisado**: Convertimos la serie en problema de regresión y usamos Random Forest.

**Lección clave**: Los modelos estadísticos como ARIMA son potentes para capturar dinámicas temporales, pero los enfoques de ML con feature engineering ofrecen flexibilidad para incluir múltiples predictores y relaciones no lineales.

En el próximo notebook (NB2) aplicaremos estos conceptos a un problema real de predicción de ventas.

---
**Fin del Notebook Conceptual - Semana 10**